## Імпортуємо датасет

In [ ]:
import pandas as pd

df = pd.read_csv("./data/titanic.csv")
df.head()

## Зробимо попередню обробку

In [ ]:
import matplotlib.pyplot as plt

# Drop 'zero' columns and clean up
df = df.loc[:, ~df.columns.str.startswith('zero')]
df = df.rename(columns={"2urvived": "Survived"})

# Перевіримо пропущені значення
print(df.isnull().sum())

# Заповнимо пропущені значення медіаною
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isnull().sum())

df.head()

## Опис колонок датасету

| Колонка | Опис |
|---|---|
| **Passengerid** | Унікальний ідентифікатор пасажира |
| **Age** | Вік пасажира (у роках) |
| **Fare** | Вартість квитка |
| **Sex** | Стать (0 = чоловік, 1 = жінка) |
| **sibsp** | Кількість братів/сестер або чоловіка/дружини на борту |
| **Parch** | Кількість батьків/дітей на борту |
| **Pclass** | Клас квитка (1 = перший, 2 = другий, 3 = третій) |
| **Embarked** | Порт посадки (0 = Шербур, 1 = Квінстаун, 2 = Саутгемптон) |
| **Survived** | Цільова змінна (0 = загинув, 1 = вижив) |

## Графіки

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Кількість виживших
df["Survived"].value_counts().plot.bar(ax=axes[0, 0], color=["salmon", "skyblue"])
axes[0, 0].set_title("Кількість виживших")
axes[0, 0].set_xticklabels(["Загинув", "Вижив"], rotation=0)

# 2. Виживання за статтю
df.groupby("Sex")["Survived"].mean().plot.bar(ax=axes[0, 1], color=["steelblue", "coral"])
axes[0, 1].set_title("Частка виживших за статтю")
axes[0, 1].set_xticklabels(["Чоловік", "Жінка"], rotation=0)
axes[0, 1].set_ylabel("Частка виживших")

# 3. Виживання за класом
df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[1, 0], color=["gold", "silver", "brown"])
axes[1, 0].set_title("Частка виживших за класом")
axes[1, 0].set_xlabel("Клас пасажира")
axes[1, 0].set_ylabel("Частка виживших")

# 4. Розподіл віку за виживанням
axes[1, 1].hist(df[df["Survived"] == 0]["Age"].dropna(), bins=20, alpha=0.6, label="Загинув", color="salmon")
axes[1, 1].hist(df[df["Survived"] == 1]["Age"].dropna(), bins=20, alpha=0.6, label="Вижив", color="skyblue")
axes[1, 1].set_title("Розподіл віку за виживанням")
axes[1, 1].set_xlabel("Вік")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Розподіл вартості квитка за виживанням

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df["Survived"] == 0]["Fare"], bins=30, alpha=0.6, label="Загинув", color="salmon")
ax.hist(df[df["Survived"] == 1]["Fare"], bins=30, alpha=0.6, label="Вижив", color="skyblue")
ax.set_title("Розподіл вартості квитка за виживанням")
ax.set_xlabel("Вартість квитка")
ax.set_ylabel("Кількість пасажирів")
ax.legend()
plt.tight_layout()
plt.show()

## Розділимо дані на тренувальну та тестову вибірки

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Passengerid", "Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f"Тренувальна вибірка: {X_train.shape[0]} рядків")
print(f"Тестова вибірка: {X_test.shape[0]} рядків")

## Метрики якості моделі

| Метрика | Англійською | Опис |
|---|---|---|
| **Точність** | Accuracy | Частка правильних передбачень серед усіх. Наприклад, якщо з 100 пасажирів модель правильно вгадала долю 80 — accuracy = 0.80 |
| **Влучність** | Precision | З усіх, кого модель назвала "вижив", скільки справді вижили. Висока precision = мало хибних спрацювань |
| **Повнота** | Recall | З усіх, хто справді вижив, скільки модель знайшла. Висока recall = модель не пропускає виживших |
| **F1** | F1-score | Середнє між precision та recall. Корисно, коли важливо балансувати між ними |
| **Матриця помилок** | Confusion Matrix | Таблиця 2×2, яка показує скільки разів модель вгадала/помилилась для кожного класу |

> **Важливо:** Precision та Recall завжди конкурують між собою — це компроміс (tradeoff). Якщо модель намагається нікого не пропустити (високий recall), вона буде частіше помилятися і "знаходити" хворобу там, де її немає (низький precision). І навпаки: якщо модель дуже обережна і каже "вижив" тільки коли впевнена (високий precision), вона пропустить багато реальних випадків (низький recall). F1-score допомагає знайти баланс між ними.

---

### Запитання для роздумів

1. Уявіть, що модель визначає чи є пухлина злоякісною. Що гірше: сказати здоровій людині що вона хвора, чи пропустити хворобу? Яка метрика тут найважливіша — precision чи recall?

2. Спам-фільтр для пошти. Що гірше: пропустити спам у inbox, чи відправити важливого листа в спам? Яка метрика важливіша?

3. Якщо в датасеті 95% загиблих і 5% виживших, модель яка завжди відповідає "загинув" матиме accuracy = 0.95. Чи означає це що модель хороша? Яку метрику краще використати?

## KNN (K-найближчих сусідів)

https://www.interactive-ml.com/knn.html

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ======== Параметри (можна змінювати) ========
n_neighbors = 5           # кількість сусідів — скільки найближчих точок враховувати
weights = "uniform"       # "uniform" — всі сусіди рівні, "distance" — ближчі сусіди мають більшу вагу
metric = "minkowski"      # спосіб вимірювання відстані між точками:
                          #   "minkowski" — узагальнена відстань (за замовчуванням = euclidean)
                          #   "euclidean" — пряма лінія між точками
                          #   "manhattan" — відстань "по кварталах" (сума різниць по кожній осі)
# ==============================================

knn = KNeighborsClassifier(
    n_neighbors=n_neighbors,
    weights=weights,
    metric=metric,
)

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Загинув", "Вижив"]))

### Матриця помилок (Confusion Matrix)

Матриця помилок — це таблиця 2×2, яка показує як модель класифікувала кожен приклад:

|  | Передбачення: Загинув | Передбачення: Вижив |
|---|---|---|
| **Факт: Загинув** | True Negative (TN) — правильно визначив загиблого | False Positive (FP) — помилково назвав виживши |
| **Факт: Вижив** | False Negative (FN) — пропустив виживш | True Positive (TP) — правильно визначив виживш |

- **Діагональ** (TN та TP) — це правильні передбачення
- **Все інше** (FP та FN) — помилки моделі
- Чим більші числа на діагоналі — тим краще працює модель

In [ ]:
# Confusion Matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (KNN)")
plt.tight_layout()
plt.show()

## KNN з масштабуванням (Pipeline)

Ознаки мають різні масштаби: наприклад, `Fare` може бути від 0 до 500, а `Sex` — лише 0 або 1.
KNN вимірює відстань між точками, тому більші числа домінують. `StandardScaler` приводить усі ознаки до однакового масштабу.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ======== Параметри (можна змінювати) ========
n_neighbors = 5           # кількість сусідів — скільки найближчих точок враховувати
weights = "uniform"       # "uniform" — всі сусіди рівні, "distance" — ближчі сусіди мають більшу вагу
metric = "minkowski"      # спосіб вимірювання відстані між точками:
                          #   "minkowski" — узагальнена відстань (за замовчуванням = euclidean)
                          #   "euclidean" — пряма лінія між точками
                          #   "manhattan" — відстань "по кварталах" (сума різниць по кожній осі)
# ==============================================

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        metric=metric,
    )),
])

knn_pipeline.fit(X_train, y_train)
y_pred_scaled = knn_pipeline.predict(X_test)

print(f"Accuracy (без масштабування): {accuracy_score(y_test, y_pred):.4f}")
print(f"Accuracy (з масштабуванням):  {accuracy_score(y_test, y_pred_scaled):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_scaled, target_names=["Загинув", "Вижив"]))

In [ ]:
# Confusion Matrix (KNN з масштабуванням)
cm_scaled = confusion_matrix(y_test, y_pred_scaled)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_scaled, annot=True, fmt="d", cmap="Blues", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (KNN з масштабуванням)")
plt.tight_layout()
plt.show()

### Пошук найкращих параметрів (GridSearchCV)

Замість того щоб вручну підбирати параметри, можна перебрати всі комбінації автоматично. `GridSearchCV` перевіряє кожну комбінацію параметрів і обирає найкращу.

In [ ]:
from sklearn.model_selection import GridSearchCV

# ======== Параметри для перебору (можна змінювати) ========
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}
scoring = "accuracy"      # метрика для оптимізації:
                          #   "accuracy"           — загальна точність
                          #   "f1"                 — F1 для класу "Вижив"
                          #   "f1_macro"           — F1 середнє обох класів (рівна вага)
                          #   "f1_weighted"        — F1 середнє обох класів (зважене за кількістю)
                          #   "precision"          — precision для класу "Вижив"
                          #   "precision_macro"    — precision середнє обох класів
                          #   "precision_weighted" — precision зважене обох класів
                          #   "recall"             — recall для класу "Вижив"
                          #   "recall_macro"       — recall середнє обох класів
                          #   "recall_weighted"    — recall зважене обох класів
# ==========================================================

grid_search = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring=scoring,
)

grid_search.fit(X_train, y_train)

print(f"Scoring: {scoring}")
print(f"Найкращі параметри: {grid_search.best_params_}")
print(f"Найкращий {scoring} (крос-валідація): {grid_search.best_score_:.4f}")
print()

y_pred_best = grid_search.predict(X_test)
print(f"Accuracy на тестовій вибірці: {accuracy_score(y_test, y_pred_best):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=["Загинув", "Вижив"]))

In [ ]:
# Порівняння KNN: до та після GridSearchCV
print("=== KNN ДО оптимізації ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_scaled):.4f}")
print(classification_report(y_test, y_pred_scaled, target_names=["Загинув", "Вижив"]))

print("=== KNN ПІСЛЯ оптимізації (GridSearchCV) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(classification_report(y_test, y_pred_best, target_names=["Загинув", "Вижив"]))

# Confusion Matrix (KNN після GridSearchCV)
cm_knn_best = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_knn_best, annot=True, fmt="d", cmap="Blues", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (KNN після GridSearchCV)")
plt.tight_layout()
plt.show()

## Дерево рішень (Decision Tree)

https://www.interactive-ml.com/decision-tree.html

Дерево рішень працює як серія запитань: "Чи стать = жінка?", "Чи вік < 30?", і так далі. На кожному кроці дерево обирає ознаку, яка найкраще розділяє дані на групи.

**Важливо:** Дерева рішень **не потребують масштабування** — вони працюють з порогами ("вік < 30"), а не з відстанями між точками.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# ======== Параметри (можна змінювати) ========
max_depth = None          # максимальна глибина дерева (None = без обмежень, або число: 3, 5, 10...)
min_samples_split = 2     # мінімальна кількість прикладів у вузлі, щоб він міг розділитися далі
                          #   якщо у групі менше прикладів — вона стає фінальною відповіддю (листком)
min_samples_leaf = 1      # мінімальна кількість прикладів у кожному листку (фінальній групі)
                          #   якщо розділення створить групу з меншою кількістю — воно заборонене
                          # Обидва параметри контролюють перенавчання (overfitting):
                          #   більші значення = простіше дерево, менше перенавчання
criterion = "gini"        # критерій розділення — як дерево вирішує яке питання задати:
                          #   "gini" — вимірює "забрудненість" групи (чим більше змішані класи, тим гірше)
                          #   "entropy" — вимірює невизначеність (чим важче вгадати клас, тим гірше)
                          #   на практиці результати схожі, але entropy працює трохи повільніше
# ==============================================

dt = DecisionTreeClassifier(
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    criterion=criterion,
    random_state=42,
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=["Загинув", "Вижив"]))

In [ ]:
# Confusion Matrix (Decision Tree)
cm_dt = confusion_matrix(y_test, y_pred_dt)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Greens", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (Decision Tree)")
plt.tight_layout()
plt.show()

### Візуалізація дерева рішень

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt, feature_names=X.columns, class_names=["Загинув", "Вижив"],
          filled=True, rounded=True, max_depth=3, ax=ax)
ax.set_title("Дерево рішень (показано перші 3 рівні)")
plt.tight_layout()
plt.show()

### Пошук найкращих параметрів (GridSearchCV)

In [ ]:
# ======== Параметри для перебору (можна змінювати) ========
param_grid_dt = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"],
}
scoring_dt = "accuracy"   # метрика для оптимізації:
                          #   "accuracy", "f1", "f1_macro", "f1_weighted"
                          #   "precision", "precision_macro", "precision_weighted"
                          #   "recall", "recall_macro", "recall_weighted"
# ==========================================================

grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=5,
    scoring=scoring_dt,
)

grid_search_dt.fit(X_train, y_train)

print(f"Scoring: {scoring_dt}")
print(f"Найкращі параметри: {grid_search_dt.best_params_}")
print(f"Найкращий {scoring_dt} (крос-валідація): {grid_search_dt.best_score_:.4f}")
print()

y_pred_dt_best = grid_search_dt.predict(X_test)
print(f"Accuracy на тестовій вибірці: {accuracy_score(y_test, y_pred_dt_best):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_dt_best, target_names=["Загинув", "Вижив"]))

In [ ]:
# Порівняння Decision Tree: до та після GridSearchCV
print("=== Decision Tree ДО оптимізації ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt, target_names=["Загинув", "Вижив"]))

print("=== Decision Tree ПІСЛЯ оптимізації (GridSearchCV) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt_best):.4f}")
print(classification_report(y_test, y_pred_dt_best, target_names=["Загинув", "Вижив"]))

# Confusion Matrix (Decision Tree після GridSearchCV)
cm_dt_best = confusion_matrix(y_test, y_pred_dt_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_dt_best, annot=True, fmt="d", cmap="Greens", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (Decision Tree після GridSearchCV)")
plt.tight_layout()
plt.show()

## Логістична регресія (Logistic Regression)

https://www.interactive-ml.com/logistic-regression.html

Логістична регресія обчислює ймовірність того, що пасажир вижив (від 0 до 1). Якщо ймовірність > 0.5 — модель відповідає "вижив", інакше — "загинув".

**Важливо:** Логістична регресія **потребує масштабування**, тому використовуємо Pipeline зі StandardScaler.

In [ ]:
from sklearn.linear_model import LogisticRegression

# ======== Параметри (можна змінювати) ========
C = 1.0                   # сила регуляризації (менше значення = сильніша регуляризація)
                          #   регуляризація не дає моделі "запам'ятовувати" тренувальні дані
                          #   спробуйте: 0.01, 0.1, 1.0, 10, 100
solver = "lbfgs"          # алгоритм оптимізації:
                          #   "lbfgs" — стандартний, підходить для більшості задач
                          #   "liblinear" — краще для маленьких датасетів
max_iter = 200            # максимальна кількість ітерацій (якщо модель не збігається — збільшіть)
# ==============================================

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=C,
        solver=solver,
        max_iter=max_iter,
        random_state=42,
    )),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["Загинув", "Вижив"]))

Accuracy: 0.7481

Classification Report:
              precision    recall  f1-score   support

     Загинув       0.78      0.92      0.84       190
       Вижив       0.58      0.31      0.40        72

    accuracy                           0.75       262
   macro avg       0.68      0.61      0.62       262
weighted avg       0.72      0.75      0.72       262



In [ ]:
# Confusion Matrix (Logistic Regression)
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Oranges", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (Logistic Regression)")
plt.tight_layout()
plt.show()

### Пошук найкращих параметрів (GridSearchCV)

In [ ]:
# ======== Параметри для перебору (можна змінювати) ========
param_grid_lr = {
    "lr__C": [0.01, 0.1, 1.0, 10, 100],
    "lr__solver": ["lbfgs", "liblinear"],
}
scoring_lr = "accuracy"   # метрика для оптимізації:
                          #   "accuracy", "f1", "f1_macro", "f1_weighted"
                          #   "precision", "precision_macro", "precision_weighted"
                          #   "recall", "recall_macro", "recall_weighted"
# ==========================================================

grid_search_lr = GridSearchCV(
    lr_pipeline,
    param_grid_lr,
    cv=5,
    scoring=scoring_lr,
)

grid_search_lr.fit(X_train, y_train)

print(f"Scoring: {scoring_lr}")
print(f"Найкращі параметри: {grid_search_lr.best_params_}")
print(f"Найкращий {scoring_lr} (крос-валідація): {grid_search_lr.best_score_:.4f}")
print()

y_pred_lr_best = grid_search_lr.predict(X_test)
print(f"Accuracy на тестовій вибірці: {accuracy_score(y_test, y_pred_lr_best):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_lr_best, target_names=["Загинув", "Вижив"]))

In [ ]:
# Порівняння Logistic Regression: до та після GridSearchCV
print("=== Logistic Regression ДО оптимізації ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=["Загинув", "Вижив"]))

print("=== Logistic Regression ПІСЛЯ оптимізації (GridSearchCV) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr_best):.4f}")
print(classification_report(y_test, y_pred_lr_best, target_names=["Загинув", "Вижив"]))

# Confusion Matrix (Logistic Regression після GridSearchCV)
cm_lr_best = confusion_matrix(y_test, y_pred_lr_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr_best, annot=True, fmt="d", cmap="Oranges", xticklabels=["Загинув", "Вижив"], yticklabels=["Загинув", "Вижив"], ax=ax)
ax.set_xlabel("Передбачення")
ax.set_ylabel("Факт")
ax.set_title("Матриця помилок (Logistic Regression після GridSearchCV)")
plt.tight_layout()
plt.show()

## Порівняння всіх моделей

Перевіримо всі моделі з різними параметрами та знайдемо найкращу комбінацію.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# ======== Scoring (можна змінювати) ========
scoring = "f1_macro"      # "accuracy", "f1", "f1_macro", "precision", "recall" тощо
# ===========================================

models = {
    "KNN": {
        "pipeline": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["euclidean", "manhattan"],
        },
    },
    "Decision Tree": {
        "pipeline": Pipeline([("model", DecisionTreeClassifier(random_state=42))]),
        "params": {
            "model__max_depth": [3, 5, 7, 10, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 5],
            "model__criterion": ["gini", "entropy"],
        },
    },
    "Logistic Regression": {
        "pipeline": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(random_state=42, max_iter=200))]),
        "params": {
            "model__C": [0.01, 0.1, 1.0, 10, 100],
            "model__solver": ["lbfgs", "liblinear"],
        },
    },
    "SVM": {
        "pipeline": Pipeline([("scaler", StandardScaler()), ("model", SVC(random_state=42))]),
        "params": {
            "model__C": [0.1, 1.0, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"],
        },
    },
    "Naive Bayes": {
        "pipeline": Pipeline([("scaler", StandardScaler()), ("model", GaussianNB())]),
        "params": {
            "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6],
        },
    },
}

results = []

for name, config in models.items():
    print(f"Тренуємо {name}...")
    grid = GridSearchCV(
        config["pipeline"],
        config["params"],
        cv=5,
        scoring=scoring,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    # Форматуємо параметри без префіксу "model__"
    best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}

    results.append({
        "Модель": name,
        f"Найкращий {scoring} (CV)": round(grid.best_score_, 4),
        "Accuracy (тест)": round(test_accuracy, 4),
        "Параметри": str(best_params),
    })

print("Готово!\n")

# Таблиця результатів
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(f"Найкращий {scoring} (CV)", ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1
results_df.index.name = "Місце"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
results_df